In [1]:
# Fit notebook to browser window
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
import ROOT
import math 
import time
import os
import numpy as np
from array import array
import uproot
import pandas as pd

In [28]:
# Get input TTree
#dir_base = '/eos/home-l/lportale/VBFparkingNtuples/merged/'
dir_base = ''
dir_in = dir_base
DFs = {}

file_in = {
#    "B" : "/eos/home-l/lportale/VBFparkingNtuples/histos_24H.root", #dir_in + 'histos_24B_PNetTau_240722_v2.root',
#    "C" : "/eos/home-l/lportale/VBFparkingNtuples/histos_24H.root", #dir_in + 'histos_24C_PNetTau_240722_v2.root',
#    "D" : "/eos/home-l/lportale/VBFparkingNtuples/histos_24H.root", #dir_in + 'histos_24D_PNetTau_240722_v2.root',
#    "E" : "/eos/home-l/lportale/VBFparkingNtuples/histos_24H.root", #dir_in + 'histos_24E_PNetTau_240722_v2.root',
#    "F" : "/eos/home-l/lportale/VBFparkingNtuples/histos_24H.root", #dir_in + 'histos_24F_PNetTau_240722_v2.root',
    #"25BC" : "/eos/home-l/lportale/VBFparkingNtuples/250606_1b1tau/histos_25BC.root"
    "25CD" : "/eos/home-l/lportale/VBFparkingNtuples/histos_Run25CD_250730.root",
    #"25D" : "/eos/home-l/lportale/VBFparkingNtuples/histos_25D_250729.root"

}
df = {}
for key in file_in:
    df[key] = ROOT.RDataFrame('VBFNanoAOD/VBFtree',file_in[key])

runBC     ='run>=379786 && run<380266'
runDpre   ='run>=380266 && run<380647'
runDEpost ='run>=380647'


In [29]:
for key in file_in:
    print(key, df[key].Count().GetValue())

25CD 151437874


In [30]:
df["25CD"].Describe()

Dataframe from TChain VBFNanoAOD/VBFtree in file /eos/home-l/lportale/VBFparkingNtuples/histos_Run25CD_250730.root

Property                Value
--------                -----
Columns in total          128
Columns from defines        0
Event loops run             1
Processing slots            1

Column                  Type                            Origin
------                  ----                            ------
BX                      Float_t                         Dataset
HLT_ele_eta             ROOT::VecOps::RVec<Float_t>     Dataset
HLT_ele_id              ROOT::VecOps::RVec<Float_t>     Dataset
HLT_ele_l1pt            ROOT::VecOps::RVec<Float_t>     Dataset
HLT_ele_phi             ROOT::VecOps::RVec<Float_t>     Dataset
HLT_ele_pt              ROOT::VecOps::RVec<Float_t>     Dataset
HLT_jets_eta            ROOT::VecOps::RVec<Float_t>     Dataset
HLT_jets_id             ROOT::VecOps::RVec<Float_t>     Dataset
HLT_jets_l1pt           ROOT::VecOps::RVec<Float_t>     Dataset
H

In [31]:
def fill_eff(h_num, h_den, year,var,version):
    eff = ROOT.TGraphAsymmErrors(h_num[year][var][version].GetValue(),
                                 h_den[year][var].GetValue(),
                                 'eff_{}'.format(var))
    return eff

def redrawBorder():
    ROOT.gPad.Update()
    ROOT.gPad.RedrawAxis()
    l = ROOT.TLine()
    l.SetLineWidth(2)
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymax(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymax())
    l.DrawLine(ROOT.gPad.GetUxmin(), ROOT.gPad.GetUymin(), ROOT.gPad.GetUxmax(), ROOT.gPad.GetUymin())

def print_unique_runs(df):
    df_runs = df.AsNumpy(["run"])
    np.unique(df_runs['run'])
    for run in np.unique(df_runs['run']):
        nevts = df.Filter('run=={}'.format(run)).Count().GetValue()
        if nevts!=0:
            print(int(run))

In [32]:
# at least 4 b/tau jet candidates
ROOT.gInterpreter.Declare('''
bool passptreqs(
              ROOT::VecOps::RVec<Float_t> pt,
              ROOT::VecOps::RVec<Float_t> eta,
              ROOT::VecOps::RVec<Float_t> id) {
    int njet = 0;
    const auto size = pt.size();
    for (size_t idx1=0; idx1<size; ++idx1){
     //if(fabs(eta[idx1])<2.5 and pt[idx1]>20. and id[idx1]>=1) njet++;
     if(fabs(eta[idx1])<2.5 and pt[idx1]>20.) njet++;

    }
    return (njet>3);
}
''')

# evaluate HT from central jets (|eta| < 2.5) with pt > 25 GeV
ROOT.gInterpreter.Declare('''
float get_HT(
                 ROOT::VecOps::RVec<Float_t> pt,
                 ROOT::VecOps::RVec<Float_t> eta,
                 ROOT::VecOps::RVec<Float_t> id) {
    float HT = 0;
    const auto size = pt.size();
    for (size_t idx1=0; idx1<size; ++idx1){
     //if(std::fabs(eta[idx1])<2.5 and pt[idx1]>25. and id[idx1]>=6) HT+=pt[idx1];
     if(std::fabs(eta[idx1])<2.5 and pt[idx1]>25. and id[idx1]==1) HT+=pt[idx1];
    }
    return HT;
}
''')

# has a btagged jet (pT>30 GeV)
ROOT.gInterpreter.Declare('''
float nbjets(
                 ROOT::VecOps::RVec<Float_t> pt,
                 ROOT::VecOps::RVec<Float_t> eta,
                 ROOT::VecOps::RVec<Float_t> id,
                 ROOT::VecOps::RVec<Float_t> pnetb) {
    const auto size = pt.size();
    float nb=0;
    for (size_t idx1=0; idx1<size; ++idx1){
     if(std::fabs(eta[idx1])<2.5 and pt[idx1]>25. and id[idx1]==1 and pnetb[idx1]>0.1919 /* medium WP*/) nb++;
    }
    return nb;
}
''')


# highest PNet tau score
ROOT.gInterpreter.Declare('''
float get_bestPNetb(
                 ROOT::VecOps::RVec<Float_t> pt,
                 ROOT::VecOps::RVec<Float_t> eta,
                 ROOT::VecOps::RVec<Float_t> id,
                 ROOT::VecOps::RVec<Float_t> pnetb) {
    float bestpnet = -1;
    const auto size = pt.size();
    for (size_t idx1=0; idx1<size; ++idx1){
      if(fabs(eta[idx1])<2.5 and pt[idx1]>25. and pnetb[idx1]>bestpnet and id[idx1]==1)
        bestpnet=pnetb[idx1];
    }
    return bestpnet;
}
''')


# highest PNet tau score
ROOT.gInterpreter.Declare('''
float get_bestPNetTau(
                 ROOT::VecOps::RVec<Float_t> pt,
                 ROOT::VecOps::RVec<Float_t> eta,
                 ROOT::VecOps::RVec<Float_t> pnet_jet,
                 ROOT::VecOps::RVec<Float_t> pnet_ele,
                 ROOT::VecOps::RVec<Float_t> pnet_muo) {
    float bestpnet = -1;
    const auto size = pt.size();
    for (size_t idx1=0; idx1<size; ++idx1){
      if(fabs(eta[idx1])<2.3 and pt[idx1]>25. and pnet_jet[idx1]>bestpnet and pnet_ele[idx1]>0.4 and pnet_muo[idx1]>0.958)
        bestpnet=pnet_jet[idx1];
    }
    return bestpnet;
}
''')

# highest PNet tau score - removing jets overlapping with good muons
ROOT.gInterpreter.Declare('''
float get_bestPNetTau_muonVeto(
                 ROOT::VecOps::RVec<Float_t> pt,
                 ROOT::VecOps::RVec<Float_t> eta,
                 ROOT::VecOps::RVec<Float_t> phi,
                 ROOT::VecOps::RVec<Float_t> id,
                 ROOT::VecOps::RVec<Float_t> pnet,
                 ROOT::VecOps::RVec<Float_t> mu_pt,
                 ROOT::VecOps::RVec<Float_t> mu_eta,
                 ROOT::VecOps::RVec<Float_t> mu_phi
) {
    float bestpnet = -1;
    
    TLorentzVector mu(mu_pt[0],mu_eta[0],mu_phi[0],0);
    
    const auto size = pt.size();
    for (size_t idx1=0; idx1<size; ++idx1){
      TLorentzVector jet(pt[idx1],eta[idx1],phi[idx1],0);
      if(fabs(eta[idx1])<2.5 
         and pt[idx1]>25. 
         and id[idx1]>=6 
         and pnet[idx1]>bestpnet) { 
         //and mu.DeltaR(jet)>0.5) {
         //std::cout<< mu.DeltaR(jet)<<std::endl;
         bestpnet=pnet[idx1];
        }
    }
    return bestpnet;
}

''')

# idx of jet with highest PNet tau score
ROOT.gInterpreter.Declare('''
float get_bestPNetTauIdx(
                 ROOT::VecOps::RVec<Float_t> pt,
                 ROOT::VecOps::RVec<Float_t> eta,
                 ROOT::VecOps::RVec<Float_t> pnet_jet,
                 ROOT::VecOps::RVec<Float_t> pnet_ele,
                 ROOT::VecOps::RVec<Float_t> pnet_muo) {
    float bestpnet = -1;
    int idx=-1;
    const auto size = pt.size();
    for (size_t idx1=0; idx1<size; ++idx1){
      // selection for tauh objects
      if(fabs(eta[idx1])<2.3 and pt[idx1]>25. and pnet_jet[idx1]>bestpnet and pnet_ele[idx1]>0.4 and pnet_muo[idx1]>0.958){
          idx = idx1;
          bestpnet=pnet_jet[idx1];
        }
    }
    return idx;
}
''')

# DR between two objects
ROOT.gInterpreter.Declare('''
float get_DR(Float_t eta1, Float_t phi1, Float_t eta2, Float_t phi2) {

    float deta2 = (eta1 - eta2)*(eta1-eta2);
    float dphi2 = (phi1 - phi2)*(phi1-phi2);
    return sqrt(deta2 + dphi2);

}

'''
)

#matching:
ROOT.gInterpreter.Declare('''
bool checkbit(int var, int pos){
  return (var & (1<<pos));
}

bool isTauMatched(
                 Float_t pt,
                 Float_t eta,
                 Float_t phi,
                 ROOT::VecOps::RVec<Float_t> hlt_pt,
                 ROOT::VecOps::RVec<Float_t> hlt_eta,
                 ROOT::VecOps::RVec<Float_t> hlt_phi,
                 ROOT::VecOps::RVec<Int_t> hlt_filter){

  bool isMatched = false;               

  const auto size = hlt_pt.size();
  for (size_t idx1=0; idx1<size; ++idx1){
    Float_t Deta2 = (hlt_eta[idx1] - eta) * (hlt_eta[idx1] - eta);
    Float_t Dphi2 = (hlt_phi[idx1] - phi) * (hlt_phi[idx1] - phi);
    Float_t DR = sqrt(Deta2 + Dphi2);
    if (DR<0.3) continue;
    
    // pass PNet loose from POG
    if(checkbit(hlt_filter[idx1],0) && checkbit(hlt_filter[idx1],4)) isMatched = true;
    
  }
  return isMatched;
}
''')

False

input_line_324:2:6: error: redefinition of 'passptreqs'
bool passptreqs(
     ^
input_line_93:2:6: note: previous definition is here
bool passptreqs(
     ^
input_line_325:2:7: error: redefinition of 'get_HT'
float get_HT(
      ^
input_line_94:2:7: note: previous definition is here
float get_HT(
      ^
input_line_326:2:7: error: redefinition of 'nbjets'
float nbjets(
      ^
input_line_95:2:7: note: previous definition is here
float nbjets(
      ^
input_line_327:2:7: error: redefinition of 'get_bestPNetb'
float get_bestPNetb(
      ^
input_line_96:2:7: note: previous definition is here
float get_bestPNetb(
      ^
input_line_328:2:7: error: redefinition of 'get_bestPNetTau'
float get_bestPNetTau(
      ^
input_line_97:2:7: note: previous definition is here
float get_bestPNetTau(
      ^
input_line_329:2:7: error: redefinition of 'get_bestPNetTau_muonVeto'
float get_bestPNetTau_muonVeto(
      ^
input_line_98:2:7: note: previous definition is here
float get_bestPNetTau_muonVeto(
    

In [33]:
ROOT.gInterpreter.Declare('''

int getNgoodTaus(ROOT::VecOps::RVec<Float_t> pt,
                 ROOT::VecOps::RVec<Float_t> eta,
                 ROOT::VecOps::RVec<Float_t> pnet_jet,
                 ROOT::VecOps::RVec<Float_t> pnet_ele,
                 ROOT::VecOps::RVec<Float_t> pnet_muo){
    int ntau=0;
    const auto size = pt.size();
    for (size_t idx1=0; idx1<size; ++idx1){
          if(fabs(eta[idx1])<2.3 and pt[idx1]>25. and pnet_jet[idx1]>0.1 and pnet_ele[idx1]>0.2 and pnet_muo[idx1]>0.2)
          ntau++;
    }
    return ntau;
}
''')

False

input_line_333:3:5: error: redefinition of 'getNgoodTaus'
int getNgoodTaus(ROOT::VecOps::RVec<Float_t> pt,
    ^
input_line_102:3:5: note: previous definition is here
int getNgoodTaus(ROOT::VecOps::RVec<Float_t> pt,
    ^


In [9]:
# highest mjj among all jet pairs (with pT>60 GeV, id tight) + additional variables for VBF jets
ROOT.gInterpreter.Declare('''
#include <math.h>       /* tanh, atanh */
''')

True

In [34]:
#periods = ["B","C","D","E","F"]
periods = ["25CD"]
df_mjj_all = {}

# apply basic filter & evaluate variables
for year in periods:
    
    cut_df = df[year]    
    
    df_mjj_all[year] = cut_df\
                    .Define('pass_pt', 'passptreqs(jets_pt,jets_eta,jets_id)')\
                    .Define('HT', 'get_HT(jets_pt,jets_eta,jets_id)')\
                    .Define('pnetbest', 'get_bestPNetTau(tau_pt,tau_eta,tau_PNETidJet,tau_PNETidEle,tau_PNETidMuon)')\
                    .Define('pnetbbest', 'get_bestPNetb(jets_pt,jets_eta,jets_id,jets_btagPNetB)')\
                    .Define('ngoodtaus', 'getNgoodTaus(tau_pt,tau_eta,tau_PNETidJet,tau_PNETidEle,tau_PNETidMuon)')\
                    .Define('bestidx',  'get_bestPNetTauIdx(jets_pt,jets_eta,tau_PNETidJet,tau_PNETidEle,tau_PNETidMuon)')\
                    .Define('DRmutau', 'get_DR(tau_eta[bestidx],tau_phi[bestidx],mu_eta[0],mu_phi[0])')\
                    .Define('atan_pnetbest', 'atanh(pnetbest)')\
                    .Define('nbjet', 'nbjets(jets_pt,jets_eta,jets_id,jets_btagPNetB)')\
                    .Define('isTauMatched','isTauMatched(tau_pt[bestidx],tau_eta[bestidx],tau_phi[bestidx],HLT_tau_pt,HLT_tau_eta,HLT_tau_phi,HLT_tau_id)')

In [11]:
test = df_mjj_all["25BC"].Histo1D(('test','',20,0,2),'DRmutau')

In [12]:
def print_unique_runs2(df):
    df_runs = df.AsNumpy(["run"])
    np.unique(df_runs['run'])
    for run in np.unique(df_runs['run']):
        nevts = df.Filter('run=={} && (passHHparking_tauref2)'.format(run)).Count().GetValue()
        if nevts==0:
            print(int(run))
            
#print_unique_runs2(DFs['DEpost'])

In [13]:
veto_runs = []

In [35]:
df_den = {}
df_num = {}
df_base = {}

for year in periods:
    print('Filling dfs for {}'.format(year))

    ## For eff vs PNet
    #df_base[year] = df_mjj_all[year].Filter('passHHparking_tauref2')\
    #                                .Filter('pass_pt')\
    #                                .Filter('HT>250')\
    #                                .Filter('nmu==1')\
    #                                .Filter('nele==0')\
    #                                .Filter('ngoodtaus==1')\
    #                                .Filter('std::fabs(mu_eta[0])<2.1')\
    #                                .Filter('DRmutau>0.4')\
    #                                .Filter('tau_PNETidEle[0]>0.4 and tau_PNETidMuon[0]>0.958 && fabs(tau_eta[0])<2.3')#\
    

    
    ## for eff vs HT
#    df_base[year] = df_mjj_all[year].Filter('pass_pt')\ # 4 jets, pT > 20 GeV, |eta|<2.5, tight ID
#                                    .Filter('nmu==1')\ # single muon (tight ID pT > 26) GeV
#                                    .Filter('nele==0')\ # no electron
#                                    .Filter('nbjet>1')\ # 2 b-tagged jets, medium WP
#                                    .Filter('ngoodtaus==1')\ #exactly good 1 tauh
#                                    .Filter('std::fabs(mu_eta[0])<2.1')\ # muon is central
#                                    .Filter('DRmutau>0.4')\ # muon well separated from tau
#                                    .Filter('tau_PNETidEle[0]>0.4 and tau_PNETidMuon[0]>0.958 && fabs(tau_eta[0])<2.3 && tau_PNETidJet[0]>0.881')#\
#                                    .Filter('isTauMatched')

    df_base[year] = df_mjj_all[year].Filter('pass_pt')\
                                    .Filter('nmu==1')\
                                    .Filter('nele==0')\
                                    .Filter('nbjet>1')\
                                    .Filter('ngoodtaus==1')\
                                    .Filter('std::fabs(mu_eta[0])<2.1')\
                                    .Filter('DRmutau>0.4')\
                                    .Filter('tau_PNETidEle[0]>0.4 and tau_PNETidMuon[0]>0.958 && fabs(tau_eta[0])<2.3 && tau_PNETidJet[0]>0.881')#\
#                                    .Filter('isTauMatched')

    df_base[year].Snapshot("VBFNanoAOD", "/eos/home-l/lportale/VBFparkingNtuples/snapshots/histos_{}.snapshot_1b1tau.root".format(year))

Filling dfs for 25CD


In [15]:
ROOT.gInterpreter.Declare('''
float get_L1HT(
                 ROOT::VecOps::RVec<Float_t> pt,
                 ROOT::VecOps::RVec<Float_t> eta) {
    float HT = 0;
    const auto size = pt.size();
    for (size_t idx1=0; idx1<size; ++idx1){
     //if(std::fabs(eta[idx1])<2.5 and pt[idx1]>25. and id[idx1]>=6) HT+=pt[idx1];
     if(std::fabs(eta[idx1])<2.5 and pt[idx1]>30) HT+=pt[idx1];
    }
    return HT;
}
''')

True

In [18]:
df_base = {}
df_den = {}
df_num = {}
periods = ["25BC","25D"]

df_base["25BC"] = ROOT.RDataFrame("VBFNanoAOD", "/eos/home-l/lportale/VBFparkingNtuples/snapshots/histos_25BC.snapshot_loose_pt.root")
df_base["25D"] = ROOT.RDataFrame("VBFNanoAOD", "/eos/home-l/lportale/VBFparkingNtuples/snapshots/histos_25D.snapshot_loose_pt.root")


In [19]:
#df_base[year].Snapshot("VBFNanoAOD",dir_in+f'/ttree_cut_{year}.root')
    
    #for run in veto_runs:
    #    df_base[year] = df_base[year].Filter('run!={}'.format(run))
for year in periods:

    df_den[year] = {}
    df_den[year]['pnet'] = df_base[year]
    df_den[year]['pnet25'] = df_base[year]
    
    df_num[year] = { 'pnet' : {} , 'pnet25' : {}, 'l1ht280' : {}, 'ditau' : {}, 'l1or' : {}, 'l1orht' : {}, 'ditauht' : {}}
    df_num[year]['pnet']['new']  = df_den[year]['pnet'].Filter('passHHparking_tau')
    df_num[year]['pnet25']['new']  = df_den[year]['pnet25'].Filter('passHHparking_tau25')
    
    df_num[year]['l1ht280']['new']  = df_den[year]['pnet25'].Filter('passL1HT280')
    df_num[year]['ditau']['new']  = df_den[year]['pnet25'].Filter('passL1DiTau or passL1DiTauJet')
    df_num[year]['l1or']['new']  = df_den[year]['pnet25'].Filter('passL1HT280 or passL1DiTau or passL1DiTauJet')
    df_num[year]['l1orht']['new']  = df_den[year]['pnet25'].Filter('passL1HT280 or ((passL1DiTau or passL1DiTauJet) and get_L1HT(L1_jets_pt,L1_jets_eta)>200)')
    df_num[year]['ditauht']['new']  = df_den[year]['pnet25'].Filter('(passL1DiTau or passL1DiTauJet) and get_L1HT(L1_jets_pt,L1_jets_eta)>200')
    
#hist from DF    
h_den, h_num = {}, {}
nbin=20
for year in periods:
    print('Filling histograms for {}'.format(year))    
    h_num[year] = {'pnet' : {}, 'pnet25' : {}, 'l1ht280' : {}, 'ditau' : {}, 'l1or' : {}, 'l1orht' : {}, 'ditauht' : {}}
    h_den[year] = {}
    
    h_den[year]['pnet'] = df_den[year]['pnet'].Histo1D(    ('h_den','',   nbin,0,600),'HT')
    h_num[year]['pnet'] = {
        'new': df_num[year]['pnet']['new'].Histo1D(    ('h_num_new','',   nbin,0,600),'HT')
    }    
    
    h_den[year]['pnet25'] = df_den[year]['pnet25'].Histo1D(    ('h_den','',   nbin,0,600),'HT')
    h_num[year]['pnet25'] = {
        'new': df_num[year]['pnet25']['new'].Histo1D(    ('h_num_new','',   nbin,0,600),'HT')
    }    
        
    h_den[year]['l1ht280'] = df_den[year]['pnet25'].Histo1D(    ('h_den','',   nbin,0,600),'HT')
    h_num[year]['l1ht280'] = {
        'new': df_num[year]['l1ht280']['new'].Histo1D(    ('h_num_new','',   nbin,0,600),'HT')
    }    
    
    
    h_den[year]['ditau'] = df_den[year]['pnet25'].Histo1D(    ('h_den','',   nbin,0,600),'HT')
    h_num[year]['ditau'] = {
        'new': df_num[year]['ditau']['new'].Histo1D(    ('h_num_new','',   nbin,0,600),'HT')
    }    
        
    h_den[year]['l1or'] = df_den[year]['pnet25'].Histo1D(    ('h_den','',   nbin,0,600),'HT')
    h_num[year]['l1or'] = {
        'new': df_num[year]['l1or']['new'].Histo1D(    ('h_num_new','',   nbin,0,600),'HT')
    }    
        
    h_den[year]['l1orht'] = df_den[year]['pnet25'].Histo1D(    ('h_den','',   nbin,0,600),'HT')
    h_num[year]['l1orht'] = {
        'new': df_num[year]['l1orht']['new'].Histo1D(    ('h_num_new','',   nbin,0,600),'HT')
    }    

    h_den[year]['ditauht'] = df_den[year]['pnet25'].Histo1D(    ('h_den','',   nbin,0,600),'HT')
    h_num[year]['ditauht'] = {
        'new': df_num[year]['ditauht']['new'].Histo1D(    ('h_num_new','',   nbin,0,600),'HT')
    }    


print("all histo filled")
 

Filling histograms for 25BC
Filling histograms for 25D
all histo filled


In [20]:
#print("{num}/{den}".format(den=h_den["25BC"]["pnet25"]["new"].Integral(),
#                       num=h_num["25BC"]["pnet25"]["new"].Integral()))

In [21]:
#h_num["25BC"]["pnet25"]["new"].Integral()

In [22]:
#periods = ['23all','23mc','23C']
#periods = ['B','C','D','E','F']
#periods = ['24B']

Eff = {}
for year in periods:
    print('Evaluating efficiencies for {}'.format(year))
    Eff[year] = {'pnet' : {}, 'pnet25': {}, 'l1ht280' : {}, 'ditau' : {}, 'l1or' : {}, 'l1orht' : {}, 'ditauht' : {}}#, 'pt1' : {}, 'pt2' : {},
                   #'mjjraw' : {}, 'pt1raw' : {}, 'pt2raw' : {}}
    
    Eff[year]['pnet']['new']    = fill_eff(h_num, h_den, year, 'pnet', 'new')
    Eff[year]['pnet25']['new']  = fill_eff(h_num, h_den, year, 'pnet25', 'new')
    Eff[year]['l1ht280']['new'] = fill_eff(h_num, h_den, year, 'l1ht280', 'new')
    Eff[year]['ditau']['new']   = fill_eff(h_num, h_den, year, 'ditau', 'new')
    Eff[year]['l1or']['new']    = fill_eff(h_num, h_den, year, 'l1or', 'new')
    Eff[year]['l1orht']['new']    = fill_eff(h_num, h_den, year, 'l1orht', 'new')
    Eff[year]['ditauht']['new']    = fill_eff(h_num, h_den, year, 'ditauht', 'new')


Evaluating efficiencies for 25BC
Evaluating efficiencies for 25D


Warning in <TGraphAsymmErrors::Divide>: Number of graph points is different than histogram bins - 1 points have been skipped
Warning in <TGraphAsymmErrors::Divide>: Number of graph points is different than histogram bins - 1 points have been skipped
Warning in <TGraphAsymmErrors::Divide>: Number of graph points is different than histogram bins - 1 points have been skipped
Warning in <TGraphAsymmErrors::Divide>: Number of graph points is different than histogram bins - 1 points have been skipped
Warning in <TGraphAsymmErrors::Divide>: Number of graph points is different than histogram bins - 1 points have been skipped
Warning in <TGraphAsymmErrors::Divide>: Number of graph points is different than histogram bins - 1 points have been skipped
Warning in <TGraphAsymmErrors::Divide>: Number of graph points is different than histogram bins - 1 points have been skipped
Warning in <TGraphAsymmErrors::Divide>: Number of graph points is different than histogram bins - 1 points have been skipped


In [23]:
# Defining variables, categories & cuts

jet_vars = ['pnet','pt1','pt2','mjj','detajj','chf']

add_cut_label = {
    'pnet'    : ' ',
}

# Plotting labels
VARLABELS = {
    #'pnet'   :  ' Transformed PNet #tau-tag score ',
    'pnet'      : 'H_{T} [GeV]',
    'pnetraw'   :  ' PNet #tau-tag score (Vs Jet) '
}

import CMS_lumi, tdrstyle
import array

tdrstyle.setTDRStyle()

#change the CMS_lumi variables (see CMS_lumi.py)
CMS_lumi.writeExtraText = True
CMS_lumi.extraText = "  Preliminary"
#CMS_lumi.lumi_sqrtS = "22.1 fb^{-1}, 2023 (13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)
CMS_lumi.lumi_sqrtS = "2024(C+D+E+F) (13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)

iPeriod = 0
iPos = 0#11
H_ref = 600; 
W_ref = 600; 
W = W_ref
H = H_ref
# references for T, B, L, R
T = 0.08*H_ref
B = 0.14*H_ref 
L = 0.12*W_ref
R = 0.04*W_ref



In [24]:
### #######################################

CMS_lumi.lumi_sqrtS = "2025 C(13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)
theVar = 'pnet'

path = {}
canvas = {}
hpx = {}
legend = {}

text_coord = {
    'pnet':    [0.4,0.3,2.9,0.8], 
    'pnetraw':    [0.1,0.2,0.9,0.6], 
    'mjj':    [0.4,0.04,0.9,0.57],   
}

Eff['25BC'][theVar]['new'].SetLineColor(46)
Eff['25BC'][theVar]['new'].SetMarkerColor(46)
Eff['25BC'][theVar]['new'].SetMarkerSize(2)
Eff['25BC'][theVar]['new'].SetLineWidth(3)
Eff['25BC'][theVar]['new'].SetMarkerStyle(23)   

path['25BC'] = ROOT.TPaveText(text_coord[theVar][0],
                              text_coord[theVar][1],
                              text_coord[theVar][2],
                              text_coord[theVar][3])

path['25BC'].SetTextAlign(12)
path['25BC'].SetFillStyle(0)
path['25BC'].SetBorderSize(0)

path['25BC'].AddText('Offline selection: ')
path['25BC'].AddText('1 #mu (tight), p_{T}> 26 GeV')
path['25BC'].AddText('1 #tau_{h}, p_{T} > 25 GeV, PNetVsMu(e) > 0.2')
path['25BC'].AddText('#Delta R (#mu, #tau) > 0.5')

c=ROOT.TCanvas('bazeqqb',"Trigger Efficicency",1280,1024)
c.SetFillColor(0)
c.SetBorderMode(0)
c.SetFrameFillStyle(0)
c.SetFrameBorderMode(0)
c.SetLeftMargin( L/W )
c.SetRightMargin( R/W )
c.SetTopMargin( T/H )
c.SetBottomMargin( B/H )
c.SetTickx()
c.SetTicky()
c.cd()

#dummy hist for consistent display
xlow=Eff['25BC'][theVar]['new'].GetXaxis().GetBinLowEdge(0)
xhigh=Eff['25BC'][theVar]['new'].GetXaxis().GetBinUpEdge(92)
hpx['25BC'] = ROOT.TH2F("hpx","",10,xlow,xhigh,10,-0.02,1.2)
hpx['25BC'].SetStats(False)
hpx['25BC'].SetTitle(theVar)
hpx['25BC'].GetXaxis().SetTitle(VARLABELS[theVar])
hpx['25BC'].GetYaxis().SetTitle("Trigger Efficiency")
hpx['25BC'].GetXaxis().SetTitleSize(0.055)
hpx['25BC'].GetXaxis().SetTitleOffset(1.1)
hpx['25BC'].GetYaxis().SetTitleSize(0.06)
hpx['25BC'].GetYaxis().SetTitleOffset(0.9)
hpx['25BC'].Draw()

legend['25BC'] = ROOT.TLegend(0.15,0.73,0.5,0.88) # top left
legend['25BC'].SetTextSize(0.04)
legend['25BC'].SetFillStyle(0)
legend['25BC'].SetBorderSize(0)
#legend['25BC'].AddEntry(Eff['B'][theVar]['new'],     'Era B',                 'lp')
#legend['25BC'].AddEntry(Eff['C'][theVar]['new'],     'Era C',                 'lp')
#legend['25BC'].AddEntry(Eff['25BC'][theVar]['new'],     'Era D',                 'lp')
#legend['25BC'].AddEntry(Eff['E'][theVar]['new'], 'Era E', 'lp')
legend['25BC'].AddEntry(Eff['25BC']['pnet25']['new'],   '2025 path',      'lp')
legend['25BC'].AddEntry(Eff['25BC']['pnet']['new'],   '2024 path',      'lp')

#path['25BC'].SetTextSize(0.045);

#Eff['B'][theVar]['new'].Draw("p same")
#Eff['C'][theVar]['new'].Draw("p same")
#Eff['25BC'][theVar]['new'].Draw("p same")
#Eff['25BC'][theVar]['new'].Draw("p same")
#Eff['E'][theVar]['new'].Draw("p same")
Eff['25BC']['pnet']['new'].Draw("p same")
Eff['25BC']['pnet25']['new'].Draw("p same")

legend['25BC'].Draw("same")
path['25BC'].Draw("same")
c.Update()
CMS_lumi.CMS_lumi(c, iPeriod, iPos)

redrawBorder()

#canvas[theVar].Draw()
c.Draw()


c.SaveAs('/eos/home-l/lportale/testcanvas.png')
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.pdf".format(theVar))
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.png".format(theVar))
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.root".format(theVar))
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.C".format(theVar))


Info in <TCanvas::Print>: png file /eos/home-l/lportale/testcanvas.png has been created


In [25]:
### #######################################

CMS_lumi.lumi_sqrtS = "2025 (13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)
theVar = 'pnet'

path = {}
canvas = {}
hpx = {}
legend = {}

text_coord = {
    'pnet':    [0.4,0.3,2.9,0.8], 
    'pnetraw':    [0.1,0.2,0.9,0.6], 
    'mjj':    [0.4,0.04,0.9,0.57],   
}

Eff['25BC']['l1or']['new'].SetLineColor(46)
Eff['25BC']['l1or']['new'].SetMarkerColor(46)
Eff['25BC']['l1or']['new'].SetMarkerSize(2)
Eff['25BC']['l1or']['new'].SetLineWidth(3)
Eff['25BC']['l1or']['new'].SetMarkerStyle(23)   


Eff['25BC']['l1ht280']['new'].SetLineColor(9)
Eff['25BC']['l1ht280']['new'].SetMarkerColor(9)
Eff['25BC']['l1ht280']['new'].SetMarkerSize(2)
Eff['25BC']['l1ht280']['new'].SetLineWidth(3)
Eff['25BC']['l1ht280']['new'].SetMarkerStyle(23)   


Eff['25BC']['ditau']['new'].SetLineColor(8)
Eff['25BC']['ditau']['new'].SetMarkerColor(8)
Eff['25BC']['ditau']['new'].SetMarkerSize(2)
Eff['25BC']['ditau']['new'].SetLineWidth(3)
Eff['25BC']['ditau']['new'].SetMarkerStyle(23)   

Eff['25BC']['l1orht']['new'].SetLineColor(6)
Eff['25BC']['l1orht']['new'].SetMarkerColor(6)
Eff['25BC']['l1orht']['new'].SetMarkerSize(2)
Eff['25BC']['l1orht']['new'].SetLineWidth(3)
Eff['25BC']['l1orht']['new'].SetMarkerStyle(23)   

path['25BC'] = ROOT.TPaveText(text_coord[theVar][0],
                              text_coord[theVar][1],
                              text_coord[theVar][2],
                              text_coord[theVar][3])

path['25BC'].SetTextAlign(12)
path['25BC'].SetFillStyle(0)
path['25BC'].SetBorderSize(0)

path['25BC'].AddText('Offline selection: ')
path['25BC'].AddText('1 #mu (tight), p_{T}> 26 GeV')
path['25BC'].AddText('1 #tau_{h}, p_{T} > 25 GeV, PNetVsMu(e) > 0.2')
path['25BC'].AddText('#Delta R (#mu, #tau) > 0.5')

c=ROOT.TCanvas('bazeazeb',"Trigger Efficicency",1280,1024)
c.SetFillColor(0)
c.SetBorderMode(0)
c.SetFrameFillStyle(0)
c.SetFrameBorderMode(0)
c.SetLeftMargin( L/W )
c.SetRightMargin( R/W )
c.SetTopMargin( T/H )
c.SetBottomMargin( B/H )
c.SetTickx()
c.SetTicky()
c.cd()

#dummy hist for consistent display
xlow=Eff['25BC'][theVar]['new'].GetXaxis().GetBinLowEdge(0)
xhigh=Eff['25BC'][theVar]['new'].GetXaxis().GetBinUpEdge(92)
hpx['25BC'] = ROOT.TH2F("hpx","",10,xlow,xhigh,10,-0.02,1.2)
hpx['25BC'].SetStats(False)
hpx['25BC'].SetTitle(theVar)
hpx['25BC'].GetXaxis().SetTitle(VARLABELS[theVar])
hpx['25BC'].GetYaxis().SetTitle("L1T Efficiency")
hpx['25BC'].GetXaxis().SetTitleSize(0.055)
hpx['25BC'].GetXaxis().SetTitleOffset(1.1)
hpx['25BC'].GetYaxis().SetTitleSize(0.06)
hpx['25BC'].GetYaxis().SetTitleOffset(0.9)
hpx['25BC'].Draw()

legend['25BC'] = ROOT.TLegend(0.15,0.73,0.5,0.88) # top left
legend['25BC'].SetTextSize(0.04)
legend['25BC'].SetFillStyle(0)
legend['25BC'].SetBorderSize(0)
#legend['25BC'].AddEntry(Eff['B'][theVar]['new'],     'Era B',                 'lp')
#legend['25BC'].AddEntry(Eff['C'][theVar]['new'],     'Era C',                 'lp')
#legend['25BC'].AddEntry(Eff['25BC'][theVar]['new'],     'Era D',                 'lp')
#legend['25BC'].AddEntry(Eff['E'][theVar]['new'], 'Era E', 'lp')
legend['25BC'].AddEntry(Eff['25BC']['l1orht']['new'],   'L1HTT280er OR (Ditau(+jet) AND L1HT>200)',      'lp')
legend['25BC'].AddEntry(Eff['25BC']['l1or']['new'],   'L1HTT280er OR Ditau(+jet)',      'lp')
legend['25BC'].AddEntry(Eff['25BC']['l1ht280']['new'],   'L1HTT280er',      'lp')
legend['25BC'].AddEntry(Eff['25BC']['ditau']['new'],   'Ditau(+jet)',      'lp')

#path['25BC'].SetTextSize(0.045);

#Eff['B'][theVar]['new'].Draw("p same")
#Eff['C'][theVar]['new'].Draw("p same")
#Eff['25BC'][theVar]['new'].Draw("p same")
#Eff['25BC'][theVar]['new'].Draw("p same")
#Eff['E'][theVar]['new'].Draw("p same")
Eff['25BC']['l1orht']['new'].Draw("p same")
Eff['25BC']['l1or']['new'].Draw("p same")
Eff['25BC']['l1ht280']['new'].Draw("p same")
Eff['25BC']['ditau']['new'].Draw("p same")

legend['25BC'].Draw("same")
path['25BC'].Draw("same")
c.Update()
CMS_lumi.CMS_lumi(c, iPeriod, iPos)

redrawBorder()

#canvas[theVar].Draw()
c.Draw()


c.SaveAs('/eos/home-l/lportale/testcanvas.png')
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.pdf".format(theVar))
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.png".format(theVar))
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.root".format(theVar))
#canvas['25BC'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.C".format(theVar))


Info in <TCanvas::Print>: png file /eos/home-l/lportale/testcanvas.png has been created


In [26]:
### #######################################

CMS_lumi.lumi_sqrtS = "2025 (13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)
theVar = 'pnet'

path = {}
canvas = {}
hpx = {}
legend = {}

text_coord = {
    'pnet':    [0.4,0.3,2.9,0.8], 
    'pnetraw':    [0.1,0.2,0.9,0.6], 
    'mjj':    [0.4,0.04,0.9,0.57],   
}

Eff['25D']['l1or']['new'].SetLineColor(46)
Eff['25D']['l1or']['new'].SetMarkerColor(46)
Eff['25D']['l1or']['new'].SetMarkerSize(2)
Eff['25D']['l1or']['new'].SetLineWidth(3)
Eff['25D']['l1or']['new'].SetMarkerStyle(23)   


Eff['25D']['l1ht280']['new'].SetLineColor(9)
Eff['25D']['l1ht280']['new'].SetMarkerColor(9)
Eff['25D']['l1ht280']['new'].SetMarkerSize(2)
Eff['25D']['l1ht280']['new'].SetLineWidth(3)
Eff['25D']['l1ht280']['new'].SetMarkerStyle(23)   


Eff['25D']['ditau']['new'].SetLineColor(8)
Eff['25D']['ditau']['new'].SetMarkerColor(8)
Eff['25D']['ditau']['new'].SetMarkerSize(2)
Eff['25D']['ditau']['new'].SetLineWidth(3)
Eff['25D']['ditau']['new'].SetMarkerStyle(23)   

Eff['25D']['l1orht']['new'].SetLineColor(6)
Eff['25D']['l1orht']['new'].SetMarkerColor(6)
Eff['25D']['l1orht']['new'].SetMarkerSize(2)
Eff['25D']['l1orht']['new'].SetLineWidth(3)
Eff['25D']['l1orht']['new'].SetMarkerStyle(23)   

path['25D'] = ROOT.TPaveText(text_coord[theVar][0],
                              text_coord[theVar][1],
                              text_coord[theVar][2],
                              text_coord[theVar][3])

path['25D'].SetTextAlign(12)
path['25D'].SetFillStyle(0)
path['25D'].SetBorderSize(0)

path['25D'].AddText('Offline selection: ')
path['25D'].AddText('1 #mu (tight), p_{T}> 26 GeV')
path['25D'].AddText('1 #tau_{h}, p_{T} > 25 GeV, PNetVsMu(e) > 0.2')
path['25D'].AddText('#Delta R (#mu, #tau) > 0.5')

c=ROOT.TCanvas('bazeazeb',"Trigger Efficicency",1280,1024)
c.SetFillColor(0)
c.SetBorderMode(0)
c.SetFrameFillStyle(0)
c.SetFrameBorderMode(0)
c.SetLeftMargin( L/W )
c.SetRightMargin( R/W )
c.SetTopMargin( T/H )
c.SetBottomMargin( B/H )
c.SetTickx()
c.SetTicky()
c.cd()

#dummy hist for consistent display
xlow=Eff['25D'][theVar]['new'].GetXaxis().GetBinLowEdge(0)
xhigh=Eff['25D'][theVar]['new'].GetXaxis().GetBinUpEdge(92)
hpx['25D'] = ROOT.TH2F("hpx","",10,xlow,xhigh,10,-0.02,1.2)
hpx['25D'].SetStats(False)
hpx['25D'].SetTitle(theVar)
hpx['25D'].GetXaxis().SetTitle(VARLABELS[theVar])
hpx['25D'].GetYaxis().SetTitle("L1T Efficiency")
hpx['25D'].GetXaxis().SetTitleSize(0.055)
hpx['25D'].GetXaxis().SetTitleOffset(1.1)
hpx['25D'].GetYaxis().SetTitleSize(0.06)
hpx['25D'].GetYaxis().SetTitleOffset(0.9)
hpx['25D'].Draw()

legend['25D'] = ROOT.TLegend(0.15,0.73,0.5,0.88) # top left
legend['25D'].SetTextSize(0.04)
legend['25D'].SetFillStyle(0)
legend['25D'].SetBorderSize(0)
#legend['25D'].AddEntry(Eff['B'][theVar]['new'],     'Era B',                 'lp')
#legend['25D'].AddEntry(Eff['C'][theVar]['new'],     'Era C',                 'lp')
#legend['25D'].AddEntry(Eff['25D'][theVar]['new'],     'Era D',                 'lp')
#legend['25D'].AddEntry(Eff['E'][theVar]['new'], 'Era E', 'lp')
legend['25D'].AddEntry(Eff['25D']['l1orht']['new'],   'L1HTT280er OR (Ditau(+jet) AND L1HT>200)',      'lp')
legend['25D'].AddEntry(Eff['25D']['l1or']['new'],   'L1HTT280er OR Ditau(+jet)',      'lp')
legend['25D'].AddEntry(Eff['25D']['l1ht280']['new'],   'L1HTT280er',      'lp')
legend['25D'].AddEntry(Eff['25D']['ditau']['new'],   'Ditau(+jet)',      'lp')

#path['25D'].SetTextSize(0.045);

#Eff['B'][theVar]['new'].Draw("p same")
#Eff['C'][theVar]['new'].Draw("p same")
#Eff['25D'][theVar]['new'].Draw("p same")
#Eff['25D'][theVar]['new'].Draw("p same")
#Eff['E'][theVar]['new'].Draw("p same")
Eff['25D']['l1orht']['new'].Draw("p same")
Eff['25D']['l1or']['new'].Draw("p same")
Eff['25D']['l1ht280']['new'].Draw("p same")
Eff['25D']['ditau']['new'].Draw("p same")

legend['25D'].Draw("same")
path['25D'].Draw("same")
c.Update()
CMS_lumi.CMS_lumi(c, iPeriod, iPos)

redrawBorder()

#canvas[theVar].Draw()
c.Draw()


c.SaveAs('/eos/home-l/lportale/testcanvas.png')
#canvas['25D'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.pdf".format(theVar))
#canvas['25D'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.png".format(theVar))
#canvas['25D'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.root".format(theVar))
#canvas['25D'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.C".format(theVar))


Warning in <TCanvas::Constructor>: Deleting canvas with same name: bazeazeb
Info in <TCanvas::Print>: png file /eos/home-l/lportale/testcanvas.png has been created


In [27]:
### #######################################

CMS_lumi.lumi_sqrtS = "2025 D(13.6 TeV)" # used with iPeriod = 0, e.g. for simulation-only plots (default is an empty string)
theVar = 'pnet'

path = {}
canvas = {}
hpx = {}
legend = {}

text_coord = {
    'pnet':    [0.4,0.3,2.9,0.8], 
    'pnetraw':    [0.1,0.2,0.9,0.6], 
    'mjj':    [0.4,0.04,0.9,0.57],   
}

Eff['25D'][theVar]['new'].SetLineColor(46)
Eff['25D'][theVar]['new'].SetMarkerColor(46)
Eff['25D'][theVar]['new'].SetMarkerSize(2)
Eff['25D'][theVar]['new'].SetLineWidth(3)
Eff['25D'][theVar]['new'].SetMarkerStyle(23)   

path['25D'] = ROOT.TPaveText(text_coord[theVar][0],
                              text_coord[theVar][1],
                              text_coord[theVar][2],
                              text_coord[theVar][3])

path['25D'].SetTextAlign(12)
path['25D'].SetFillStyle(0)
path['25D'].SetBorderSize(0)

path['25D'].AddText('Offline selection: ')
path['25D'].AddText('1 #mu (tight), p_{T}> 26 GeV')
path['25D'].AddText('1 #tau_{h}, p_{T} > 25 GeV, PNetVsMu(e) > 0.2')
path['25D'].AddText('#Delta R (#mu, #tau) > 0.5')

c=ROOT.TCanvas('bazaaaeb',"Trigger Efficicency",1280,1024)
c.SetFillColor(0)
c.SetBorderMode(0)
c.SetFrameFillStyle(0)
c.SetFrameBorderMode(0)
c.SetLeftMargin( L/W )
c.SetRightMargin( R/W )
c.SetTopMargin( T/H )
c.SetBottomMargin( B/H )
c.SetTickx()
c.SetTicky()
c.cd()

#dummy hist for consistent display
xlow=Eff['25D'][theVar]['new'].GetXaxis().GetBinLowEdge(0)
xhigh=Eff['25D'][theVar]['new'].GetXaxis().GetBinUpEdge(92)
hpx['25D'] = ROOT.TH2F("hpx","",10,xlow,xhigh,10,-0.02,1.2)
hpx['25D'].SetStats(False)
hpx['25D'].SetTitle(theVar)
hpx['25D'].GetXaxis().SetTitle(VARLABELS[theVar])
hpx['25D'].GetYaxis().SetTitle("Trigger Efficiency")
hpx['25D'].GetXaxis().SetTitleSize(0.055)
hpx['25D'].GetXaxis().SetTitleOffset(1.1)
hpx['25D'].GetYaxis().SetTitleSize(0.06)
hpx['25D'].GetYaxis().SetTitleOffset(0.9)
hpx['25D'].Draw()

legend['25D'] = ROOT.TLegend(0.15,0.73,0.5,0.88) # top left
legend['25D'].SetTextSize(0.04)
legend['25D'].SetFillStyle(0)
legend['25D'].SetBorderSize(0)
#legend['25D'].AddEntry(Eff['B'][theVar]['new'],     'Era B',                 'lp')
#legend['25D'].AddEntry(Eff['C'][theVar]['new'],     'Era C',                 'lp')
#legend['25D'].AddEntry(Eff['25D'][theVar]['new'],     'Era D',                 'lp')
#legend['25D'].AddEntry(Eff['E'][theVar]['new'], 'Era E', 'lp')
legend['25D'].AddEntry(Eff['25D']['pnet25']['new'],   '2025 path',      'lp')
legend['25D'].AddEntry(Eff['25D']['pnet']['new'],   '2024 path',      'lp')

#path['25D'].SetTextSize(0.045);

#Eff['B'][theVar]['new'].Draw("p same")
#Eff['C'][theVar]['new'].Draw("p same")
#Eff['25D'][theVar]['new'].Draw("p same")
#Eff['25D'][theVar]['new'].Draw("p same")
#Eff['E'][theVar]['new'].Draw("p same")
Eff['25D']['pnet']['new'].Draw("p same")
Eff['25D']['pnet25']['new'].Draw("p same")

legend['25D'].Draw("same")
path['25D'].Draw("same")
c.Update()
CMS_lumi.CMS_lumi(c, iPeriod, iPos)

redrawBorder()

#canvas[theVar].Draw()
c.Draw()


c.SaveAs('/eos/home-l/lportale/testcanvas.png')
#canvas['25D'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.pdf".format(theVar))
#canvas['25D'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.png".format(theVar))
#canvas['25D'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.root".format(theVar))
#canvas['25D'].SaveAs(dir_in + "/plots/HHpnettau_BtoE_{}.STEAM_20240621.test.C".format(theVar))


Info in <TCanvas::Print>: png file /eos/home-l/lportale/testcanvas.png has been created


In [ ]:
print(dir_in+"/plots/")

In [ ]:
ctest = ROOT.TCanvas("test","test")
ctest.cd()


# hlt histo first to "zoom in"
h_num["F"]["pnet"]['new'].SetLineColor(2)
h_num["F"]["pnet"]['new'].Draw("hist e")

# then denominator
h_den["F"]["pnet"].Draw("hist e same")




#ctest.SetLogy()
ctest.SetGridy()
ctest.SetLogy()
ctest.Draw()